# Drone Delivery Route Optimization with Deep Reinforcement Learning

> **University:** Universidade do Vale do Rio dos Sinos (UNISINOS)<br />
> **Program:** Pós-graduação em Inteligência Artificial Aplicada<br />
> **Course:** Aprendizado por Reforço<br />
> **Author:** Augusto Stahlschmidt e Eduardo Moraes Macedo<br />
> **Date:** October 5, 2025   
---

## 1. Introduction & Objectives

**Problem:**  
Autonomous delivery systems require agents capable of navigating complex environments, reaching multiple destinations in an efficient order, and returning to a base. This project investigates whether a deep Reinforcement Learning agent can learn such a policy from scratch in a simulated 2D grid world.

**Task Type:**  
Sequential decision-making under sparse and shaped rewards (Deep Reinforcement Learning).

**Business or Research Objective:**  
Train and compare three state-of-the-art RL algorithms (PPO, A2C, DQN) on a custom drone delivery task. Identify which algorithm produces the most efficient and reliable navigation policy.

**Analytical Goal:**  
Maximize cumulative episode reward by completing all deliveries and returning to the base in the minimum number of steps.

**Primary Evaluation Metrics:**  
- Average Reward per episode (effectiveness)  
- Average Episode Length in steps (efficiency)

**Success Criteria:**  
An agent is considered successful if it completes all deliveries and returns to base within the 150-step limit, achieving an average reward above 400.

## 2. Setup & Imports

In [4]:
import os
import gc
import time

import numpy as np
import torch as th

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.env_checker import check_env

import optuna

import pygame
from moviepy import VideoFileClip, TextClip, ImageSequenceClip
from IPython.display import display, Video

SEED = 42
np.random.seed(SEED)

DEVICE = "cuda" if th.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


## 3. Problem Formulation

The drone delivery task is modeled as a Markov Decision Process (MDP) with the following components:

**State Space (S)**

Each state is a tuple of three variables:

| Component | Description | Cardinality |
|-----------|-------------|-------------|
| `agent_location` | Current drone position (row, col) on the 6x6 grid | 36 |
| `delivery_status` | Binary vector indicating pending deliveries per house | 2^4 = 16 |
| `mission_status` | 0 = deliver mode, 1 = return-to-base mode | 2 |

Total state space: 36 x 16 x 2 = **1,152 states**.

**Action Space (A)**

Four discrete movement actions:

| Action | Direction |
|--------|----------|
| 0 | Up (row - 1) |
| 1 | Down (row + 1) |
| 2 | Left (col - 1) |
| 3 | Right (col + 1) |

**Transition Dynamics**

The agent moves to the target cell if it is (a) within grid bounds, (b) not a tree obstacle, and (c) not a house without a pending delivery. Invalid moves are rejected and incur a collision penalty.

**Reward Function**

A hybrid reward function balances exploration guidance with sparse objective rewards:

| Event | Reward |
|-------|--------|
| Each time step | -0.01 (step penalty) |
| Invalid move | -0.5 (collision penalty) |
| Approach target | +0.2 x delta distance (shaping) |
| Successful delivery | +100 |
| Return to base | +250 |
| Episode timeout | 0 (truncated) |

**Episode Termination**

- Terminated: agent returns to base after all deliveries.  
- Truncated: step count reaches the 150-step limit.

## 4. Environment Design

The simulation environment is implemented as a custom Gymnasium environment (`DroneDeliveryEnv`). It renders a 6x6 grid representing a neighborhood with roads, houses, trees, and an establishment (base).

**Fixed configuration used in all experiments:**

| Parameter | Value |
|-----------|-------|
| Grid size | 6x6 |
| Number of houses | 4 |
| Number of tree obstacles | 3 |
| Max steps per episode | 150 |
| Observation space | Dict (location + delivery status + mission mode) |
| Action space | Discrete(4) |

At the start of each episode, between 2 and 4 houses are randomly selected to receive deliveries, creating diverse scenarios across training and evaluation.

In [5]:
class DroneDeliveryEnv(gym.Env):
    """
    Custom Gymnasium environment simulating a drone performing deliveries on a 6x6 grid
    neighborhood. The agent must deliver packages to all requested houses and return to
    the establishment, while avoiding tree obstacles and houses without pending orders.
    """

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 5}

    def __init__(self, grid_size=6, n_houses=4, render_mode=None):
        super().__init__()

        self.grid_size = grid_size
        self.n_houses = n_houses
        self.render_mode = render_mode
        self.window_size = 512
        self.max_steps_per_episode = 150

        self._establishment_location = np.array([0, 0])
        self._road_locations = self._generate_roads()
        self._house_locations, self._tree_locations = self._generate_locations_in_blocks()
        self._agent_location = np.array([0, 0])

        self.observation_space = spaces.Dict({
            "agent_location": spaces.Box(0, self.grid_size - 1, shape=(2,), dtype=np.int32),
            "delivery_status": spaces.MultiBinary(self.n_houses),
            "mission_status": spaces.Discrete(2),
        })

        self.action_space = spaces.Discrete(4)

        self._action_to_direction = {
            0: np.array([-1, 0]),  # Up
            1: np.array([1, 0]),   # Down
            2: np.array([0, -1]), # Left
            3: np.array([0, 1]),  # Right
        }

        self.window = None
        self.clock = None
        if self.render_mode:
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))

    def _generate_roads(self):
        """Generate road cells in a regular grid pattern (every 2 cells)."""
        roads = set()
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if i % 2 == 0 or j % 2 == 0:
                    roads.add((i, j))
        roads.add(tuple(self._establishment_location))
        return roads

    def _generate_locations_in_blocks(self):
        """Place houses and trees in non-road cells with a fixed random seed for reproducibility."""
        rng = np.random.default_rng(123)
        occupied = self._road_locations | {tuple(self._establishment_location)}

        available = [
            np.array((r, c))
            for r in range(self.grid_size)
            for c in range(self.grid_size)
            if (r, c) not in occupied
        ]
        rng.shuffle(available)

        house_locs = available[:self.n_houses]
        tree_locs = available[self.n_houses:self.n_houses + 3]
        return np.array(house_locs), tree_locs

    def _get_obs(self):
        return {
            "agent_location": self._agent_location.astype(np.int32),
            "delivery_status": self._delivery_status,
            "mission_status": np.int32(self._mission_status),
        }

    def _get_info(self):
        if self._mission_status == 0:
            targets = self._house_locations[self._delivery_status == 1]
        else:
            targets = [self._establishment_location]
        if len(targets) == 0:
            return {"distance_to_target": 0}
        dist = np.min([np.linalg.norm(self._agent_location - t, ord=1) for t in targets])
        return {"distance_to_target": float(dist)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._agent_location = self._establishment_location.copy()

        # Randomly select 2 to n_houses delivery targets per episode
        num_deliveries = self.np_random.integers(2, self.n_houses + 1)
        delivery_indices = self.np_random.choice(self.n_houses, num_deliveries, replace=False)
        self._delivery_status = np.zeros(self.n_houses, dtype=np.int8)
        self._delivery_status[delivery_indices] = 1

        self._total_deliveries = int(np.sum(self._delivery_status))
        self._deliveries_made = 0
        self._mission_status = 0
        self.current_step = 0

        info = self._get_info()
        self._last_distance = info["distance_to_target"]
        return self._get_obs(), info

    def step(self, action):
        self.current_step += 1
        direction = self._action_to_direction[action]
        next_location = self._agent_location + direction

        reward = -0.01  # Step penalty to encourage efficiency

        in_bounds = np.all(next_location >= 0) and np.all(next_location < self.grid_size)
        is_tree = any(np.array_equal(next_location, t) for t in self._tree_locations)
        is_non_target_house = any(
            np.array_equal(next_location, self._house_locations[i])
            for i in range(self.n_houses)
            if self._delivery_status[i] == 0
        )

        if in_bounds and not is_tree and not is_non_target_house:
            self._agent_location = next_location
        else:
            reward -= 0.5  # Collision penalty

        # Distance-based reward shaping
        current_dist = self._get_info()["distance_to_target"]
        reward += (self._last_distance - current_dist) * 0.2
        self._last_distance = current_dist

        terminated = False

        if self._mission_status == 1:
            if np.array_equal(self._agent_location, self._establishment_location):
                reward += 250
                terminated = True
        else:
            for i, house_loc in enumerate(self._house_locations):
                if np.array_equal(self._agent_location, house_loc) and self._delivery_status[i] == 1:
                    self._delivery_status[i] = 0
                    self._deliveries_made += 1
                    reward += 100
                    if self._deliveries_made == self._total_deliveries:
                        self._mission_status = 1
                    self._last_distance = self._get_info()["distance_to_target"]
                    break

        truncated = self.current_step >= self.max_steps_per_episode
        return self._get_obs(), reward, terminated, truncated, self._get_info()

    def render(self):
        if self.render_mode is None:
            return None
        if self.window is None:
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((134, 184, 80))
        pix = self.window_size / self.grid_size

        for road in self._road_locations:
            pygame.draw.rect(canvas, (80, 80, 80), pygame.Rect(road[1] * pix, road[0] * pix, pix, pix))
        for x in range(self.grid_size + 1):
            pygame.draw.line(canvas, (114, 164, 60), (0, pix * x), (self.window_size, pix * x), 1)
            pygame.draw.line(canvas, (114, 164, 60), (pix * x, 0), (pix * x, self.window_size), 1)

        # Trees — dark green blocks
        for tree in self._tree_locations:
            pygame.draw.rect(canvas, (34, 100, 34),
                             pygame.Rect(tree[1] * pix + 4, tree[0] * pix + 4, pix - 8, pix - 8))

        # Establishment — gold block with border
        ex, ey = self._establishment_location[1] * pix, self._establishment_location[0] * pix
        pygame.draw.rect(canvas, (255, 200, 0), pygame.Rect(ex + 4, ey + 4, pix - 8, pix - 8))
        pygame.draw.rect(canvas, (180, 130, 0), pygame.Rect(ex + 4, ey + 4, pix - 8, pix - 8), 3)

        # Houses — orange (pending) or light-green (delivered)
        for i, loc in enumerate(self._house_locations):
            color = (230, 120, 30) if self._delivery_status[i] == 1 else (120, 200, 120)
            border = (160, 70, 0) if self._delivery_status[i] == 1 else (40, 130, 40)
            hx, hy = loc[1] * pix, loc[0] * pix
            pygame.draw.rect(canvas, color, pygame.Rect(hx + 4, hy + 4, pix - 8, pix - 8))
            pygame.draw.rect(canvas, border, pygame.Rect(hx + 4, hy + 4, pix - 8, pix - 8), 3)

        # Drone — blue circle
        cx = int(self._agent_location[1] * pix + pix / 2)
        cy = int(self._agent_location[0] * pix + pix / 2)
        pygame.draw.circle(canvas, (30, 100, 220), (cx, cy), int(pix / 2) - 6)
        pygame.draw.circle(canvas, (10, 50, 160), (cx, cy), int(pix / 2) - 6, 3)

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])

        return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None


# Validate environment compliance with Gymnasium API
env_check = DroneDeliveryEnv()
check_env(env_check, warn=True)
env_check.close()
print("Environment check passed.")

Environment check passed.


## 5. Experimental Methodology

The experimental design consists of three phases: baseline training with default hyperparameters, Optuna-based hyperparameter optimization, and a final comparative evaluation of all algorithms under identical conditions.

### 5.1 Algorithms Selected

Three algorithms from Stable-Baselines3 were evaluated, covering on-policy and off-policy paradigms:

| Algorithm | Type | Rationale |
|-----------|------|----------|
| PPO (Proximal Policy Optimization) | On-policy, Actor-Critic | Strong general-purpose baseline; clip-based update prevents destabilizing policy changes |
| A2C (Advantage Actor-Critic) | On-policy, Actor-Critic | Simpler on-policy alternative to PPO; useful for direct comparison within the same family |
| DQN (Deep Q-Network) | Off-policy, Value-based | Represents a fundamentally different learning paradigm via replay buffer and Q-value estimation |

### 5.2 Performance Metrics

| Metric | Description | Direction |
|--------|-------------|----------|
| Average Reward | Mean cumulative reward across evaluation episodes | Higher is better |
| Average Episode Length | Mean steps to complete the mission | Lower is better |
| Training Time | Wall-clock seconds to train 200,000 timesteps | Informational |

Each model is evaluated deterministically (`deterministic=True`) over 5 episodes with randomly generated delivery configurations.

## 6. Model Training

### 6.1 Baseline Experiment

All three algorithms are first trained with default hyperparameters (except DQN's buffer size, reduced to 100k to avoid memory overflow). This establishes a performance reference before optimization.

In [6]:
TOTAL_TIMESTEPS = 200_000
EVAL_EPISODES = 5

BASELINE_CONFIGS = {
    "PPO_Baseline": {"model": PPO, "policy": "MultiInputPolicy", "params": {}},
    "A2C_Baseline": {"model": A2C, "policy": "MultiInputPolicy", "params": {}},
    "DQN_Baseline": {
        "model": DQN,
        "policy": "MultiInputPolicy",
        "params": {"buffer_size": 100_000},
    },
}


def train_and_evaluate(algo_name, model_class, policy, params, total_timesteps, eval_episodes):
    """Train a model, run evaluation episodes, and save a video of the best episode."""
    # Training
    train_env = DroneDeliveryEnv(grid_size=6, n_houses=4)
    model = model_class(
        policy, train_env, verbose=0,
        tensorboard_log=f"./tensorboard/{algo_name}/",
        **params,
    )
    start = time.time()
    model.learn(total_timesteps=total_timesteps, progress_bar=True)
    training_time = time.time() - start

    model_path = f"drone_{algo_name}.zip"
    model.save(model_path)
    del model
    train_env.close()

    # Evaluation
    eval_env = DroneDeliveryEnv(grid_size=6, n_houses=4, render_mode="rgb_array")
    model = model_class.load(model_path, env=eval_env)

    episode_rewards, episode_lengths, episode_frames = [], [], []

    for _ in range(eval_episodes):
        obs, _ = eval_env.reset()
        done, ep_reward, frames = False, 0.0, []
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(int(action))
            ep_reward += reward
            frames.append(eval_env.render())
            done = terminated or truncated
        episode_rewards.append(ep_reward)
        episode_lengths.append(eval_env.current_step)
        episode_frames.append(frames)

    eval_env.close()

    # Save video of the best episode
    best_idx = int(np.argmax(episode_rewards))
    video_path = f"drone_best_{algo_name}.mp4"
    clip = ImageSequenceClip(episode_frames[best_idx], fps=5)
    clip.write_videofile(video_path, codec="libx264", logger=None)

    return {
        "avg_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "avg_length": float(np.mean(episode_lengths)),
        "training_time": training_time,
        "video_path": video_path,
    }


baseline_results = {}

for name, cfg in BASELINE_CONFIGS.items():
    print(f"\nTraining: {name}")
    result = train_and_evaluate(
        name, cfg["model"], cfg["policy"], cfg["params"],
        TOTAL_TIMESTEPS, EVAL_EPISODES,
    )
    baseline_results[name] = result
    print(f"  Avg Reward: {result['avg_reward']:.2f} | "
          f"Avg Length: {result['avg_length']:.1f} | "
          f"Training Time: {result['training_time']:.1f}s")


Training: PPO_Baseline


Output()

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Output()

  Avg Reward: 613.57 | Avg Length: 19.2 | Training Time: 542.8s

Training: A2C_Baseline


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Output()

  Avg Reward: 280.50 | Avg Length: 150.0 | Training Time: 603.8s

Training: DQN_Baseline


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


  Avg Reward: 230.37 | Avg Length: 123.2 | Training Time: 340.0s


### 6.2 Hyperparameter Optimization

Preliminary baseline training suggests that A2C and DQN may require tuned hyperparameters to achieve competitive performance. While PPO converged with default settings in our initial tests, we hypothesize that systematic optimization could improve all algorithms' efficiency and final reward.

We will employ a two‑phase Optuna search to identify optimal hyperparameters for each algorithm:

- Phase 1 (Exploration): 10 trials per algorithm, each trained for 10,000 timesteps. We will use the Tree‑structured Parzen Estimator (TPE) sampler for intelligent parameter sampling and the Median pruner to terminate unpromising trials early.
- Phase 2 (Refinement): 15 additional trials per algorithm, each trained for 30,000 timesteps. The search will be seeded with the top‑three configurations from Phase 1, focusing exploration around the most promising regions.

Trials whose running average partial reward falls below –100 will be pruned immediately to conserve computational resources.

Upon execution, the Optuna search will produce optimal hyperparameter configurations for each algorithm. These configurations will be used in the final comparative experiment.

In [7]:
EXPLORE_TIMESTEPS = 10_000
EXPLORE_TRIALS    = 10
REFINE_TIMESTEPS  = 30_000
REFINE_TRIALS     = 15
EVAL_EPISODES_PER_TRIAL = 5


def build_search_space(trial: optuna.Trial, algo_name: str):
    if algo_name == "A2C":
        return A2C, {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
            "n_steps":       trial.suggest_categorical("n_steps", [16, 32, 64, 128, 256]),
            "gamma":         trial.suggest_float("gamma", 0.9, 0.999),
            "vf_coef":       trial.suggest_float("vf_coef", 0.2, 0.8),
            "ent_coef":      trial.suggest_float("ent_coef", 1e-8, 1e-2, log=True),
        }
    elif algo_name == "DQN":
        return DQN, {
            "learning_rate":          trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
            "buffer_size":            trial.suggest_categorical("buffer_size", [50_000, 100_000]),
            "batch_size":             trial.suggest_categorical("batch_size", [32, 64, 128]),
            "gamma":                  trial.suggest_float("gamma", 0.95, 0.999),
            "exploration_fraction":   trial.suggest_float("exploration_fraction", 0.1, 0.5),
            "exploration_final_eps":  trial.suggest_float("exploration_final_eps", 0.01, 0.1),
        }
    elif algo_name == "PPO":
        return PPO, {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
            "n_steps":       trial.suggest_categorical("n_steps", [512, 1024, 2048]),
            "gamma":         trial.suggest_float("gamma", 0.98, 0.9999),
            "ent_coef":      trial.suggest_float("ent_coef", 1e-8, 1e-2, log=True),
        }
    raise ValueError(f"Unsupported algorithm: {algo_name}")


def objective(trial: optuna.Trial, algo_name: str, total_timesteps: int) -> float:
    model_class, params = build_search_space(trial, algo_name)

    train_env = DroneDeliveryEnv(grid_size=6, n_houses=4)
    model = model_class("MultiInputPolicy", train_env, verbose=0, device=DEVICE, **params)
    block = total_timesteps // 10
    partial_rewards = []

    for _ in range(10):
        model.learn(total_timesteps=block, reset_num_timesteps=False)

        eval_env = DroneDeliveryEnv(grid_size=6, n_houses=4)
        obs, _ = eval_env.reset()
        done, ep_reward = False, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(int(action))
            ep_reward += reward
            done = terminated or truncated
        eval_env.close()

        partial_rewards.append(ep_reward)
        if np.mean(partial_rewards) < -100:
            train_env.close()
            return -1000.0

    # Final multi-episode evaluation
    eval_env = DroneDeliveryEnv(grid_size=6, n_houses=4)
    final_rewards = []
    for _ in range(EVAL_EPISODES_PER_TRIAL):
        obs, _ = eval_env.reset()
        done, ep_reward = False, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(int(action))
            ep_reward += reward
            done = terminated or truncated
        final_rewards.append(ep_reward)
    eval_env.close()
    train_env.close()

    gc.collect()
    th.cuda.empty_cache()
    return float(np.mean(final_rewards))


best_params = {}

for algo in ["PPO", "A2C", "DQN"]:
    print(f"\n--- Phase 1: Exploration ({algo}) ---")
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)

    study1 = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study1.optimize(
        lambda trial, a=algo: objective(trial, a, EXPLORE_TIMESTEPS),
        n_trials=EXPLORE_TRIALS, gc_after_trial=True,
    )

    top3 = sorted(study1.trials, key=lambda t: t.value or -9999, reverse=True)[:3]

    print(f"\n--- Phase 2: Refinement ({algo}) ---")
    study2 = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    for base_trial in top3:
        study2.enqueue_trial(base_trial.params)

    study2.optimize(
        lambda trial, a=algo: objective(trial, a, REFINE_TIMESTEPS),
        n_trials=REFINE_TRIALS, gc_after_trial=True,
    )

    best_params[algo] = study2.best_trial.params
    print(f"Best params for {algo}: {best_params[algo]}")
    print(f"Best reward: {study2.best_trial.value:.2f}")

[I 2026-03-30 00:33:03,564] A new study created in memory with name: no-name-85c63589-2c2e-47e2-8238-f9c922b7f463



--- Phase 1: Exploration (PPO) ---


[I 2026-03-30 00:33:33,429] Trial 0 finished with value: -73.80000000000003 and parameters: {'learning_rate': 5.6115164153345e-05, 'n_steps': 512, 'gamma': 0.9831047709448044, 'ent_coef': 8.62913219007185e-08}. Best is trial 0 with value: -73.80000000000003.
[I 2026-03-30 00:34:04,352] Trial 1 finished with value: -54.16000000000007 and parameters: {'learning_rate': 1.3066739238053272e-05, 'n_steps': 512, 'gamma': 0.9804096314364864, 'ent_coef': 0.006598711072054081}. Best is trial 1 with value: -54.16000000000007.
[I 2026-03-30 00:34:34,742] Trial 2 finished with value: 18.980000000000054 and parameters: {'learning_rate': 0.000462258900102083, 'n_steps': 512, 'gamma': 0.9860544206348948, 'ent_coef': 1.4077923139972383e-05}. Best is trial 2 with value: 18.980000000000054.
[I 2026-03-30 00:35:03,003] Trial 3 finished with value: 48.02000000000005 and parameters: {'learning_rate': 7.309539835912905e-05, 'n_steps': 1024, 'gamma': 0.9858136785058509, 'ent_coef': 1.5782327810795563e-06}. Be


--- Phase 2: Refinement (PPO) ---


[I 2026-03-30 00:39:40,313] Trial 0 finished with value: 116.7600000000009 and parameters: {'learning_rate': 8.168455894760161e-05, 'n_steps': 512, 'gamma': 0.9917890499203547, 'ent_coef': 1.8997763474111273e-08}. Best is trial 0 with value: 116.7600000000009.
[I 2026-03-30 00:41:04,858] Trial 1 finished with value: 62.21999999999999 and parameters: {'learning_rate': 7.309539835912905e-05, 'n_steps': 1024, 'gamma': 0.9858136785058509, 'ent_coef': 1.5782327810795563e-06}. Best is trial 0 with value: 116.7600000000009.
[I 2026-03-30 00:42:28,525] Trial 2 finished with value: -13.039999999999813 and parameters: {'learning_rate': 4.066563313514796e-05, 'n_steps': 1024, 'gamma': 0.9824285608734111, 'ent_coef': 9.355380606452177e-06}. Best is trial 0 with value: 116.7600000000009.
[I 2026-03-30 00:43:49,743] Trial 3 finished with value: 116.8800000000006 and parameters: {'learning_rate': 5.989003672254293e-05, 'n_steps': 1024, 'gamma': 0.9855905967427788, 'ent_coef': 1.8037506431281856e-05}.

...

Best params for A2C: {'learning_rate': 0.0004325432427964555, 'n_steps': 16, 'gamma': 0.9118666713660346, 'vf_coef': 0.40256910284217684, 'ent_coef': 0.0045442082254883295}
Best reward: 168.68

--- Phase 1: Exploration (DQN) ---


[I 2026-03-30 01:18:31,266] Trial 0 finished with value: 119.53999999999982 and parameters: {'learning_rate': 5.6115164153345e-05, 'buffer_size': 50000, 'batch_size': 32, 'gamma': 0.9528460969962418, 'exploration_fraction': 0.4464704583099741, 'exploration_final_eps': 0.0641003510568888}. Best is trial 0 with value: 119.53999999999982.
[I 2026-03-30 01:18:47,080] Trial 1 finished with value: 185.6800000000009 and parameters: {'learning_rate': 0.0002607024758370766, 'buffer_size': 100000, 'batch_size': 32, 'gamma': 0.9589868209828182, 'exploration_fraction': 0.2216968971838151, 'exploration_final_eps': 0.05722807884690141}. Best is trial 1 with value: 185.6800000000009.
[I 2026-03-30 01:19:02,402] Trial 2 finished with value: 24.320000000000004 and parameters: {'learning_rate': 7.309539835912905e-05, 'buffer_size': 100000, 'batch_size': 128, 'gamma': 0.9723474292266348, 'exploration_fraction': 0.41407038455720546, 'exploration_final_eps': 0.02797064039425238}. Best is trial 1 with value


--- Phase 2: Refinement (DQN) ---


[I 2026-03-30 01:21:35,192] Trial 0 finished with value: 290.7639999999998 and parameters: {'learning_rate': 0.00021137059440645722, 'buffer_size': 100000, 'batch_size': 128, 'gamma': 0.9879815083446946, 'exploration_fraction': 0.4757995766256756, 'exploration_final_eps': 0.0905344615384884}. Best is trial 0 with value: 290.7639999999998.
[I 2026-03-30 01:22:18,873] Trial 1 finished with value: 210.12799999999984 and parameters: {'learning_rate': 0.0002607024758370766, 'buffer_size': 100000, 'batch_size': 32, 'gamma': 0.9589868209828182, 'exploration_fraction': 0.2216968971838151, 'exploration_final_eps': 0.05722807884690141}. Best is trial 0 with value: 290.7639999999998.
[I 2026-03-30 01:23:01,185] Trial 2 finished with value: 199.62000000000083 and parameters: {'learning_rate': 0.00017643967683381525, 'buffer_size': 50000, 'batch_size': 128, 'gamma': 0.9812403160964054, 'exploration_fraction': 0.4548850970305306, 'exploration_final_eps': 0.05249934326457544}. Best is trial 0 with va

Best params for DQN: {'learning_rate': 0.0003498227300675491, 'buffer_size': 100000, 'batch_size': 128, 'gamma': 0.9508977996804002, 'exploration_fraction': 0.13122329125142715, 'exploration_final_eps': 0.0694676251961902}
Best reward: 361.44


### 6.3 Final Comparative Experiment

Each algorithm is retrained from scratch for 200,000 timesteps using its optimized hyperparameters. All agents are evaluated deterministically over 5 randomly generated episodes. The best episode per agent is saved as a video.

In [8]:
FINAL_CONFIGS = {
    "PPO_Optimized": {
        "model": PPO,
        "policy": "MultiInputPolicy",
        "params": {
            "learning_rate": 0.0005785498689372767,
            "n_steps": 2048,
            "batch_size": 32,
            "gamma": 0.9901155533367073,
            "ent_coef": 0.006320573411606166,
        },
    },
    "A2C_Optimized": {
        "model": A2C,
        "policy": "MultiInputPolicy",
        "params": {
            "learning_rate": 0.0004325432427964555,
            "n_steps": 16,
            "gamma": 0.9118666713660346,
            "vf_coef": 0.40256910284217684,
            "ent_coef": 0.0045442082254883295,
        },
    },
    "DQN_Optimized": {
        "model": DQN,
        "policy": "MultiInputPolicy",
        "params": {
            "learning_rate": 0.0003498227300675491,
            "buffer_size": 100_000,
            "batch_size": 128,
            "gamma": 0.9508977996804002,
            "exploration_fraction": 0.13122329125142715,
            "exploration_final_eps": 0.0694676251961902,
            "train_freq": 4,
            "gradient_steps": 1,
            "learning_starts": 5000,
        },
    },
}

final_results = {}

for name, cfg in FINAL_CONFIGS.items():
    print(f"\nTraining: {name}")
    result = train_and_evaluate(
        name, cfg["model"], cfg["policy"], cfg["params"],
        TOTAL_TIMESTEPS, EVAL_EPISODES,
    )
    final_results[name] = result
    print(f"  Avg Reward: {result['avg_reward']:.2f} | "
          f"Avg Length: {result['avg_length']:.1f} | "
          f"Training Time: {result['training_time']:.1f}s")

Output()


Training: PPO_Optimized


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Output()

  Avg Reward: 448.11 | Avg Length: 43.2 | Training Time: 689.6s

Training: A2C_Optimized


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Output()

  Avg Reward: 212.06 | Avg Length: 150.0 | Training Time: 498.3s

Training: DQN_Optimized


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


  Avg Reward: 573.50 | Avg Length: 18.4 | Training Time: 365.1s


## 7. Model Evaluation

### 7.1 Quantitative Results

Each model was trained for 200,000 timesteps and evaluated deterministically over 5 episodes with randomly generated delivery configurations (2 to 4 houses per episode).

**Baseline (default hyperparameters):**

| Algorithm | Avg Reward | Avg Steps | Mission Completed | Passes Criterion (> 400) |
| :--- | :---: | :---: | :---: | :---: |
| PPO | 613.57 | 19.2 | Yes | Yes |
| A2C | 280.50 | 150.0 | No | No |
| DQN | 230.37 | 123.2 | Rarely | No |

**Optimized hyperparameters (Optuna):**

| Algorithm | Avg Reward | Avg Steps | Mission Completed | Passes Criterion (> 400) |
| :--- | :---: | :---: | :---: | :---: |
| PPO | 448.11 | 43.2 | Yes | Yes |
| A2C | 212.06 | 150.0 | No | No |
| DQN | 573.40 | 18.4 | Yes | Yes |

**Notes on the Mission Completed column:**  
An average episode length of 150.0 indicates the agent always hits the step limit without returning to base. A2C reaches this ceiling in both settings, confirming it learned to make partial deliveries but never acquired the return-to-base behavior. DQN Baseline at 123.2 average steps is deceptive: back-calculating the reward implies roughly 4 out of 5 episodes time out and only 1 terminates successfully, so it is classified as rarely completing.

**Reward consistency check:**  
With an average of 3 deliveries per episode, a clean run would yield approximately 3 x 100 + 250 = 650 before step penalties. PPO Baseline (613.57, 19.2 steps) sits just below this ceiling, consistent with occasionally drawing 2-delivery episodes. DQN Optimized (573.40, 18.4 steps) matches a similar profile. A2C rewards of ~280 are consistent with completing roughly 3 deliveries but never reaching base (3 x 100 - 150 x 0.01 = 298.50, minus collision penalties).

**PPO regression after optimization:**  
PPO performance dropped 27% in reward and episode length doubled after Optuna tuning. This is a methodological artifact: the Optuna search evaluated trials using only 10,000 (Phase 1) and 30,000 (Phase 2) timesteps, but the final models train for 200,000. The selected hyperparameters favor fast early convergence, not long-run performance. Default PPO hyperparameters, designed conservatively for general use, happen to be well-suited to this task at full training length.

### 7.2 Per-Algorithm Analysis

**PPO (Proximal Policy Optimization)**

PPO achieves the highest reward in both baseline (613.57) and optimized (448.11) settings. Counterintuitively, optimization degraded performance by 27% in reward and increased average episode length from 19.2 to 43.2 steps. This is attributed to the Optuna search being conducted at 10k–30k timesteps per trial, a fraction of the 200k used for final training. The tuned configuration converges faster in early training but does not generalize to the full training regime. Default PPO parameters remain the strongest configuration tested.

**A2C (Advantage Actor-Critic)**

A2C consistently fails to complete the mission in both settings, always reaching the 150-step limit. The positive rewards (~280 baseline, ~212 optimized) indicate the agent learns to make partial deliveries but never acquires the return-to-base behavior. Optimization slightly worsened performance, suggesting the Optuna search found no viable region in the parameter space for this task. A2C may require significantly more training timesteps or a fundamentally different reward shaping to solve the full task.

**DQN (Deep Q-Network)**

DQN shows the largest benefit from hyperparameter tuning: a 149% reward increase (230.37 to 573.40) and 85% step reduction (123.2 to 18.4). The baseline agent rarely completes the mission, with roughly 4 out of 5 evaluation episodes timing out. The optimized agent reliably completes the mission and rivals the PPO baseline in efficiency. This sensitivity to hyperparameters is characteristic of off-policy value-based methods, where learning rate, exploration schedule, and replay buffer size critically determine whether the Q-function converges to a useful policy.

### 7.3 Qualitative Analysis

The videos below show the best episode recorded for each optimized agent during evaluation. Because episodes with 2 to 4 deliveries are drawn randomly, best-episode behavior may differ from the average captured in section 7.1. Descriptions note where the two diverge.

**Video 1: PPO (Optimized)**  
The agent reliably completes the mission, but the route is noticeably less direct than what PPO Baseline produces (43.2 vs 19.2 average steps). The drone still avoids obstacles and transitions correctly between delivery mode and return mode, but takes extra steps before committing to each target. This is consistent with the optimization finding parameters that converge quickly at 10k–30k timesteps but are suboptimal at full 200k training length.

In [12]:
display(Video("drone_best_PPO_Optimized.mp4", embed=True, width=600))

**Video 2: A2C (Optimized)**  
The agent makes deliveries but never returns to base, always hitting the 150-step limit. The drone navigates toward houses and collects packages correctly in early steps, then enters repetitive loops near the end of the episode without progressing to the return objective. This confirms the quantitative finding: A2C rewards around 212 reflect partial task completion, not a solved policy.

In [13]:
display(Video("drone_best_A2C_Optimized.mp4", embed=True, width=600))

**Video 3: DQN (Optimized)**  
The DQN optimized agent is the most improved across the two experiment sets. It completes the full mission, including the return to base, in an average of 18.4 steps, matching the efficiency of PPO Baseline. The trajectory is decisive and collision-free, a marked contrast to the baseline DQN which rarely completed any episode. This demonstrates that DQN's performance on this task is almost entirely a function of hyperparameter quality, particularly the exploration schedule and replay buffer size.

In [14]:
display(Video("drone_best_DQN_Optimized.mp4", embed=True, width=600))

## 8. Conclusions & Next Steps

**Key Findings**

- PPO with default hyperparameters achieved the highest average reward (613.57) and the most efficient routing (19.2 steps), reaching approximately 94% of the theoretical reward ceiling (~650 for 4 deliveries: 4x100 + 250 minus step penalties). This indicates the policy is near-optimal within the constraints of the environment.
- DQN was the largest beneficiary of hyperparameter tuning: reward increased from 230.37 to 573.50 (149% gain) and average episode length dropped from 123.2 to 18.4 steps (85% reduction). The tuned DQN matches PPO Baseline in efficiency and surpasses PPO Optimized in both metrics.
- A2C failed to complete the mission in all configurations, consistently reaching the 150-step limit. Rewards of 280.50 (baseline) and 212.06 (optimized) are consistent with completing roughly 2 to 3 deliveries without triggering the return-to-base behavior.
- The Optuna search produced a counterproductive result for PPO: the selected configuration scored 289.63 at search time (30k timesteps) but degraded performance by 27% at full training length (200k timesteps), from 613.57 down to 448.11. The search budget was too short to reflect long-run behavior.

**Model Performance vs Success Criteria**

The success criterion was defined as average reward above 400 with mission completion within 150 steps. Three of the six trained agents met this threshold:

| Agent | Avg Reward | Avg Steps | Criterion Met |
| :--- | :---: | :---: | :---: |
| PPO Baseline | 613.57 | 19.2 | Yes |
| DQN Optimized | 573.50 | 18.4 | Yes |
| PPO Optimized | 448.11 | 43.2 | Yes |
| A2C Baseline | 280.50 | 150.0 | No |
| A2C Optimized | 212.06 | 150.0 | No |
| DQN Baseline | 230.37 | 123.2 | No |

The overall objective of training an agent capable of completing the full delivery mission was achieved by PPO and DQN. A2C did not meet the criterion under any tested configuration.

**Interpretation**

In a real-world logistics context, PPO Baseline and DQN Optimized both represent viable policies: they complete all deliveries and return to base in under 20 steps on a 6x6 grid, with near-zero wasted movement. The contrast between DQN Baseline (rarely completing the mission) and DQN Optimized (reliably completing it in 18.4 steps) illustrates that off-policy value-based methods require careful tuning of the exploration schedule and replay buffer to bridge the gap between early random exploration and a convergent policy. PPO's robustness to default settings is practically significant: it reduces the engineering effort required to deploy a working agent. A2C's failure to acquire the return-to-base behavior despite learning deliveries suggests the algorithm struggles with multi-objective sequential tasks where the second objective (return) only activates after the first (all deliveries) is complete.

**Limitations**

- The Optuna hyperparameter search was conducted at 10k (Phase 1) and 30k (Phase 2) timesteps per trial, while final training used 200k timesteps. This mismatch caused PPO to regress and means the search results cannot be taken as reliable proxies for full-training performance.
- The environment is deterministic: actions always succeed, obstacles are static, and the grid layout is fixed across episodes. A real drone operates in a stochastic world with sensor noise, wind, and dynamic obstacles.
- Evaluation used only 5 episodes per agent, which is insufficient to produce statistically stable estimates, especially given the variable number of deliveries per episode (2 to 4).
- The discrete 2D grid eliminates altitude, continuous movement, battery constraints, and payload capacity, all of which are critical in a real deployment scenario.
- A2C never solved the full task. Its failure mode was not diagnosed beyond observation: further ablation (reward shaping variants, longer training, curriculum learning) would be needed to determine whether the algorithm is fundamentally unsuited to this task or simply requires conditions not tested here.

**Next Steps**

- Re-run Optuna using the same 200k timestep budget as final training, or use a proxy metric that better correlates with long-run performance (e.g., reward at the last 20% of training rather than the mean over all timesteps).
- Increase the evaluation set to at least 50 episodes to obtain statistically meaningful averages and standard deviations per agent.
- Investigate A2C failure with a curriculum approach: start with a single delivery and progressively add more, and use reward shaping that explicitly signals the transition to return mode.
- Extend the environment to a larger grid (e.g., 10x10 with more houses) to test whether learned policies generalize beyond the training configuration.
- Introduce stochastic transitions (action failure probability, randomized obstacle placement per episode) to evaluate policy robustness.
- Explore SAC (Soft Actor-Critic) or TD3 adapted to discrete action spaces, as they offer more stable off-policy learning than DQN and may reduce the current sensitivity to hyperparameter configuration.

## 9. References

- Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms*. arXiv:1707.06347  
- Mnih, V. et al. (2015). *Human-level control through deep reinforcement learning*. Nature, 518, 529-533.  
- Mnih, V. et al. (2016). *Asynchronous Methods for Deep Reinforcement Learning*. ICML.  
- Gymnasium Documentation: https://gymnasium.farama.org/  
- Stable-Baselines3 Documentation: https://stable-baselines3.readthedocs.io/  
- Optuna Documentation: https://optuna.readthedocs.io/  